# PSELDNets × v11（core 7,200本=20h、自然頻度＋重要セルフロア、純静穏・警告のみ・複数車をcoreに統合）

事前準備（Drive `MyDrive/PSELDNets_data/`）:
- `dataset_outdoor_siren_v11.zip`（約9GB、core 7,200本）← **今回新規にアップロード**
- `dataset_outdoor_siren_v10.zip`（4.7GB）と `dataset_outdoor_siren_v10_2_add.zip`（884MB）← v10.2学習で配置済みのはず（評価専用セットに使う）

- 学習 fold1_room1 4,800 / val fold2_room1 1,200 / test fold3_room1 1,199（検品FAIL1本除外。**testは最終1回まで触らない**）
- 評価枠はv10側を共用（物理同一でビット一致のため再生成なし）: 交差点20・プローブ48・6シナリオ100・交通量60・幻覚30
- **v10.2までの「空ラベル除外セル」は廃止**。純静穏582本は設計上の教師なので、空ラベル許容preproc（セル8）で学習に載せる
- 設計の正: `md/design/v11データセット拡張_設計書_2026-07-27.md`。**データを変えたら EXP_NAME を必ず変える**
- 学習は約7時間（T4/100ep）。**切れても再実行すれば last.ckpt から自動再開**（Drive永続化）

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠ GPUがありません。ランタイム→T4 GPU を選択してください'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントと設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定 ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'
DATASET    = 'outdoor_siren_v11'    # 学習用
EVAL_DS    = 'outdoor_siren_v10'    # 評価専用セット（v11と物理同一・ビット一致）
EXP_NAME   = 'outdoor_siren_v11_run1'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
V11_ZIP = f'{DRIVE_DATA}/dataset_outdoor_siren_v11.zip'
V10_ZIP = f'{DRIVE_DATA}/dataset_outdoor_siren_v10.zip'
V10_ADD = f'{DRIVE_DATA}/dataset_outdoor_siren_v10_2_add.zip'
assert os.path.exists(V11_ZIP), '⚠ v11 zipがDriveにありません'
assert os.path.exists(V10_ZIP), '⚠ v10 zip（評価用）がDriveにありません'
assert os.path.exists(V10_ADD), '⚠ v10.2追補zip（幻覚評価room）がDriveにありません'
print(f'OK: v11 {os.path.getsize(V11_ZIP)/1e9:.2f}GB / v10 {os.path.getsize(V10_ZIP)/1e9:.2f}GB / add {os.path.getsize(V10_ADD)/1e6:.0f}MB')

## 3. リポジトリ clone（コミットpin付き=再現性固定）

In [ ]:
import os

REPO = '/content/PSELDNets'
# ローカル検証（空ラベル対策・data.py:98診断等）と同一コードのコミットに固定。
# .gitmodules のpinと同じ（upstream main先端 'Add MIT License'）
PSELDNETS_COMMIT = '8092a14866963e4dc38ee389aa146f0a2edef1bb'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既に存在: {REPO}')

os.chdir(REPO)
!git fetch -q origin
!git checkout -q {PSELDNETS_COMMIT}
head = !git rev-parse HEAD
assert head[0] == PSELDNETS_COMMIT, f'⚠ commit不一致: {head}'
print(f'CWD: {os.getcwd()} / PSELDNets @ {head[0][:12]}')

## 4. 依存インストール

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive にキャッシュ

# 事前学習ckptのハッシュ固定（正= md/audit/第5回監査への対応_2026-07-19.md）
import hashlib
CKPT_SHA256 = '813083ac938c5974a6f36ceca29ea66c0382091db5df1d6d47ece9572d5ac71b'
h = hashlib.sha256(open(CKPT, 'rb').read()).hexdigest()
assert h == CKPT_SHA256, f'⚠ ckpt SHA256不一致: {h}'
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB, sha256一致)')

## 6. データセット展開（v11約9GB＋v10評価用、20分前後）＋マニフェスト照合

v11は展開結果を**マニフェストダイジェスト**（全ファイルのname+size連結md5、正=ローカル
`out/dataset_outdoor_siren_v11/manifest_v11.csv`）と照合してから、検品FAIL 1本を除外する。
除外後の最終状態（7,199本×3）にも専用ダイジェストで完全性を保証（部分欠損のすり抜け防止）。
v10側は評価専用room（fold2系858本）だけ残して間引く（学習はv11のみのため fold1/fold3 不要）。

In [ ]:
import zipfile, os, glob, hashlib

def _dir_digest(ds, sub):
    entries = sorted((os.path.basename(p), os.path.getsize(p))
                     for p in glob.glob(f'datasets/{ds}/{sub}/*'))
    dg = hashlib.md5('\n'.join(f'{n},{s}' for n, s in entries).encode()).hexdigest()
    return dg, len(entries)

# マニフェストダイジェスト（manifest_v11.csv から算出。full=7200本 / final=FAIL除外後7199本）
V11_DIGEST_FULL = {'foa': 'a9030a52e4eba2c318b6e05389872cc3',
                   'metadata': '7610c36b3c8bee697ff4485c9c9cc997',
                   'masks': '90042e05428a5383cba3fded51d469ef'}
V11_DIGEST_FINAL = {'foa': '19d017feb2810468f3aee4be4cdee7aa',
                    'metadata': '347a54db4b0bc49485e496f5cb699d52',
                    'masks': '6c3cb53722a56d5abc7815710dd41185'}
# 検品FAIL（inspection.csv 2026-07-28確定）: 受音ゲート±3.5dBの良性の裾（ベルの
# 打撃×距離の偶然相関、-4.33dBのうち-3.29dBを打撃タイミングで説明済み）。
# 物理異常ではないが v10 mix119 と同方針で除外（fold3=1,199本）
INSPECT_FAIL = ['fold3_room1_mix0181']

if not os.path.exists(f'datasets/{DATASET}/foa'):
    with zipfile.ZipFile(V11_ZIP) as z:
        z.extractall('.')
    print('v11 unzipped')
else:
    print('v11は展開済み')

_, n11 = _dir_digest(DATASET, 'foa')
if n11 == 7200:
    for sub in ('foa', 'metadata', 'masks'):
        dg, cnt = _dir_digest(DATASET, sub)
        assert (cnt, dg) == (7200, V11_DIGEST_FULL[sub]), \
            f'⚠ {sub}: 展開結果がマニフェスト(7200)と不一致 {cnt} {dg}'
    for stem in INSPECT_FAIL:
        for sub, ext in (('foa', 'flac'), ('metadata', 'csv'), ('masks', 'csv')):
            os.remove(f'datasets/{DATASET}/{sub}/{stem}.{ext}')
    print(f'manifest(7200)照合OK -> INSPECT_FAIL除外: {INSPECT_FAIL}')
for sub in ('foa', 'metadata', 'masks'):
    dg, cnt = _dir_digest(DATASET, sub)
    assert (cnt, dg) == (7199, V11_DIGEST_FINAL[sub]), \
        f'⚠ {sub}: 最終状態がマニフェスト(7199)と不一致 {cnt} {dg}'
print('v11 manifest digest OK (7,199 x3 subdirs)')

if not os.path.exists(f'datasets/{EVAL_DS}/foa'):
    with zipfile.ZipFile(V10_ZIP) as z:
        z.extractall('.')
    with zipfile.ZipFile(V10_ADD) as z:
        z.extractall('.')
    print('v10+add unzipped')
else:
    print('v10は展開済み')

# --- v10を評価専用に間引く（fold1系・fold3を削除。冪等） ---
n10 = len(os.listdir(f'datasets/{EVAL_DS}/foa'))
if n10 > 858:
    removed = 0
    for sub in ('foa', 'metadata', 'masks'):
        for p in glob.glob(f'datasets/{EVAL_DS}/{sub}/fold1_*') + \
                 glob.glob(f'datasets/{EVAL_DS}/{sub}/fold3_*'):
            os.remove(p)
            removed += 1
    n10 = len(os.listdir(f'datasets/{EVAL_DS}/foa'))
    print(f'v10剪定: {removed}ファイル削除')
assert n10 == 858, f'⚠ v10評価セットの本数が想定外: {n10} (期待858)'
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
assert n_cls == 6
print(f'v10 eval foa: {n10} / classes: {n_cls}')

## 7. 設定ファイル（v11学習＋val推論、v10評価5種）

roomsフィルタは部分文字列マッチ。学習は v11 の [fold1_room1] のみ。

In [ ]:
def data_yaml(ds, train_rooms, valid_rooms, test_rooms):
    return f"""audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  {ds}: {train_rooms}
valid_dataset:
  {ds}: {valid_rooms}
test_dataset:
  {ds}: {test_rooms}
"""

def exp_yaml(ds, data_name):
    return f"""# @package _global_
defaults:
 - override /data: {data_name}.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: {ds}

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {{lr: 0.0003}}
  lr_scheduler:
    kwargs: {{step_size: 60}}

trainer:
  max_epochs: 100
  check_val_every_n_epoch: 5
"""

# v11: 学習本体 + val推論variant
open(f'configs/data/{DATASET}.yaml', 'w').write(
    data_yaml(DATASET, '[fold1_room1]', '[fold2_room1]', '[fold3_room1]'))
open(f'configs/experiment/{DATASET}.yaml', 'w').write(exp_yaml(DATASET, DATASET))
open(f'configs/data/{DATASET}_valinfer.yaml', 'w').write(
    data_yaml(DATASET, '[fold1_room1]', '[fold2_room1]', '[fold2_room1]'))
open(f'configs/experiment/{DATASET}_valinfer.yaml', 'w').write(
    exp_yaml(DATASET, f'{DATASET}_valinfer'))

# v10: 評価5種（学習には使わない。train/validは存在するroomを指すだけ）
EVAL_TAGS = [('scenario', '[fold2_room9]'),
             ('probe', '[fold9_room1]'),
             ('scn2', '[fold2_room4, fold2_room5, fold2_room6, fold2_room7, fold2_room8]'),
             ('v10a', '[fold8_room1]'),
             ('halluc', '[fold2_room3]')]
for tag, rooms in EVAL_TAGS:
    open(f'configs/data/{EVAL_DS}_{tag}.yaml', 'w').write(
        data_yaml(EVAL_DS, '[fold2_room1]', '[fold2_room1]', rooms))
    open(f'configs/experiment/{EVAL_DS}_{tag}.yaml', 'w').write(
        exp_yaml(EVAL_DS, f'{EVAL_DS}_{tag}'))
print('wrote configs (v11 train/valinfer + v10 eval x5)')

## 8. 前処理（**空ラベル許容版**、両データセット。初回のみ40〜50分規模）

v11の純静穏582本はラベルCSVが0バイト（設計上の教師）。素のpreprocは
EmptyDataErrorで停止するため、読み込み2関数を0バイト判定でラップした別プロセスで
実行する（pin済みPSELDNets本体は非接触。設計= v11設計書§1.5、ローカル実証済み）。
**v10.2までの除外セルはv11では使わない。**

In [ ]:
from pathlib import Path

WRAPPER = r'''# _preproc_emptyok.py (auto-generated)
import os
import runpy
import sys

sys.path.insert(0, "src")
import pandas as pd
import utils.data_utilities as du
import preproc.preprocess as pp

_load, _read = du.load_output_format_file, pd.read_csv


def _is_empty_file(path):
    try:
        return os.path.getsize(path) == 0
    except (TypeError, OSError, ValueError):
        return False


def load_ok(path, *a, **kw):
    if _is_empty_file(path):
        return {99: []}          # 最終フレームのみ・イベント0件（ゼロ行列になる）
    return _load(path, *a, **kw)


def read_ok(path, *a, **kw):
    if _is_empty_file(path):
        return pd.DataFrame([[99, 0, 0, 0, 0]])   # num_frames=100 算出専用の番兵
    return _read(path, *a, **kw)


du.load_output_format_file = load_ok
pp.load_output_format_file = load_ok   # from-import名ごと差し替え
pd.read_csv = read_ok                  # 別プロセス内のみのグローバル差し替え

sys.argv = ["src/preproc.py"] + sys.argv[1:]
runpy.run_path("src/preproc.py", run_name="__main__")
'''
Path('_preproc_emptyok.py').write_text(WRAPPER, encoding='utf-8')

# 学習プロセス用ラッパ: data.py(BaseDataset)がvalの正解ラベルを生CSVから読む経路も
# 空CSVで落ちる（data/components/data.py:98、2026-07-28にColab実地で発見）。
# 同じ0バイト判定で {99: []}=イベント0件のGT を返す（val指標への偽イベント混入なし）
WRAPPER_TRAIN = r'''# _train_emptyok.py (auto-generated)
import os
import runpy
import sys

sys.path.insert(0, "src")
import utils.data_utilities as du
import data.components.data as dcd

_load = du.load_output_format_file


def _is_empty_file(path):
    try:
        return os.path.getsize(path) == 0
    except (TypeError, OSError, ValueError):
        return False


def load_ok(path, *a, **kw):
    if _is_empty_file(path):
        return {99: []}   # イベント0件のGT
    return _load(path, *a, **kw)


du.load_output_format_file = load_ok
dcd.load_output_format_file = load_ok   # data.pyはfrom-importなので名前ごと差し替え

sys.argv = ["src/train.py"] + sys.argv[1:]
runpy.run_path("src/train.py", run_name="__main__")
'''
Path('_train_emptyok.py').write_text(WRAPPER_TRAIN, encoding='utf-8')
print('wrote _preproc_emptyok.py / _train_emptyok.py')

for ds in [DATASET, EVAL_DS]:
    idx = f'_hdf5/data/24000fs/wav/dev/{ds}_10sChunklen_10sHoplen_train.csv'
    if os.path.exists(idx):
        print(f'{ds}: 前処理済み')
    else:
        !python _preproc_emptyok.py dataset={ds}
    import pandas as pd
    print(ds, 'index rows:', len(pd.read_csv(idx, header=None)))

## 9. 最終チェック

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     '事前学習チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV (6クラス)'),
    (f'datasets/{DATASET}/foa',             'v11 FOA (7199=7200-検品FAIL1)'),
    (f'datasets/{EVAL_DS}/foa',             'v10評価 FOA (858)'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   'v11前処理インデックス'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で約7時間 / 100epoch）

- **切れても再実行すれば last.ckpt から自動再開**（Drive永続化）→ 寝る前にこのセルまで実行
- val は fold2_room1 のみ。test(fold3) はここでは一切使わない
- **データを変えて学習し直すときは必ず EXP_NAME を変えること**

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python _train_emptyok.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 学習曲線（val 抜粋）

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

## 12. 推論（6セット一括: val=v11 / 交差点・プローブ・6シナリオ・交通量・幻覚=v10をv11ckptで）

予測CSVをセットごとに1本へ連結してDriveに保存 → ローカルの解剖・採点が読む
（ファイル名はv10.2と同じ規約 `infer_{EXP_NAME}_{tag}_all.csv`）。

In [ ]:
import glob, os
cands = sorted(glob.glob('/content/drive/MyDrive/PSELDNets_logs*'))
cands += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/PSELDNets_logs'))
best_ckpt = None
for c in cands:
    hits = sorted(glob.glob(f'{c}/{DATASET}/runs/{EXP_NAME}/checkpoints/epoch_*.ckpt'))
    if hits:
        DRIVE_LOGS = c
        best_ckpt = hits[-1]
        break
print('best_ckpt =', best_ckpt)
assert best_ckpt, '⚠ ckptが見えません（学習が終わっていますか）'

JOBS = [(f'{DATASET}_valinfer', 'val'),
        (f'{EVAL_DS}_scenario', 'scenario'),
        (f'{EVAL_DS}_probe', 'probe'),
        (f'{EVAL_DS}_scn2', 'scn2'),
        (f'{EVAL_DS}_v10a', 'v10a'),
        (f'{EVAL_DS}_halluc', 'halluc')]
for experiment, short in JOBS:
    exp = f'infer_{EXP_NAME}_{short}'
    !python src/infer.py experiment={experiment} \
        mode=test \
        ckpt_path="{best_ckpt}" \
        model.kwargs.pretrained_path=null \
        experiment_name={exp} \
        paths.log_dir={DRIVE_LOGS}
    task = experiment.rsplit('_', 1)[0]
    sub = f'{DRIVE_LOGS}/{task}/runs/{exp}/submissions'
    out_lines = []
    for p in sorted(glob.glob(f'{sub}/*.csv')):
        stem = os.path.basename(p)[:-4]
        for line in open(p):
            if line.strip():
                out_lines.append(f'{stem},{line.strip()}')
    out = f'{DRIVE_DATA}/{exp}_all.csv'
    open(out, 'w').write('\n'.join(out_lines))
    print('wrote', out, len(out_lines), 'lines')

## 13. このあと（ローカル側）

1. 6つの `infer_..._all.csv` をローカルへ → 解剖・通知層採点・シナリオ採点・v10a同時検出・幻覚検定
2. fold3(test)は全分析が固まった後に**最終1回だけ**（先生と相談してから）
3. デコーダ閾値掃引（threshold_unify 5/10/15/20°）は**統合前トラック別出力の保存セル**を別途用意して実施（v11設計書§4.5）
4. 因果推論・batch=1ベンチ・傾き耐性のセルはv10.2用をパス替えで再利用可